# Delta Demo — Day 5: MERGE INTO (Deep Clone + One Atomic Transaction)
**Prerequisite:** Delta_Demo_Day1_Load and Delta_Demo_Day2_Load must already have been run — this notebook Deep Clones the `employees` table as it stood at the end of Day 2 (version 1), so Day 3's later changes don't need to exist yet.

Shows the SAME logical outcome as Day 3's four manual statements, but expressed as ONE atomic `MERGE INTO` statement against a fresh cloned copy — everything runs in one shot in this notebook: clone → build source → merge → verify.

## Day 3 (Alt) — Same Result via One Atomic MERGE INTO
Everything we did manually on Day 3 (one UPDATE, one DELETE, one more UPDATE, one INSERT = 4 separate transactions) can instead be expressed as a SINGLE `MERGE INTO` statement. We'll do this on a fresh COPY of the table (a Deep Clone taken from right after Day 2), so it doesn't disturb the Day 3 story we already recorded above.

In [0]:
%sh
# Clean up any leftover copy from a previous run, so the clone below starts fresh.
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_merge_demo

## Check whats in Employees before cloning it

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

In [0]:
%sql
-- DEEP CLONE makes a fully independent physical copy (both the log AND the data
-- files) of the table exactly as it looked at version 1 (end of Day 2).

CREATE or replace TABLE delta.`/Volumes/workspace/delta_demo/demo_files/employees_merge_demo`
DEEP CLONE delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

## Build the Day 3 Source Dataset (Changes to Merge)
MERGE needs a "source" dataset describing everything that should change. We encode each row's intent with a simple action flag: 'U' = update, 'D' = delete, 'I' = insert. This flag isn't a Delta feature — it's just a column we made up ourselves to drive the MERGE logic below.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW day3_source AS
SELECT * FROM VALUES
  (1, 'Ravi',        27500, 'U'),
  (2, NULL,          NULL,  'D'),
  (6, 'Shanmukh J',  45000, 'U'),
  (7, 'Divya',       30000, 'I')
AS day3(eno, ename, sal, action);

In [0]:
%sql
-- Sanity check: view the source dataset before merging it in.
select * from day3_source

In [0]:
%sql
-- Sanity check: confirm the cloned table still matches Day 2's state before we
-- run MERGE against it.
select * from delta.`/Volumes/workspace/delta_demo/demo_files/employees_merge_demo` order by eno;

## Run the MERGE
One statement, evaluated row by row against a join on `eno`:
- Matched + action='D' → DELETE that row
- Matched + action='U' → UPDATE that row's ename/sal
- Not matched + action='I' → INSERT it as a new row
All four logical changes commit as ONE atomic transaction — one new entry in `_delta_log`, not four.

In [0]:
%sql
MERGE INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees_merge_demo` AS t
USING day3_source AS s
ON t.eno = s.eno
WHEN MATCHED AND s.action = 'D' THEN DELETE
WHEN MATCHED AND s.action = 'U' THEN UPDATE SET t.ename = s.ename, t.sal = s.sal
WHEN NOT MATCHED AND s.action = 'I' THEN INSERT (eno, ename, sal) VALUES (s.eno, s.ename, s.sal);

/* Expected result after running the MERGE:

1  Ravi           27500.00
3  Uma Maheshwar  36999.00   (untouched)
5  Kanth          50000.00   (untouched)
6  Shanmukh J     45000.00
7  Divya          30000.00
*/


## Describe History — MERGE Commit (Compare to Manual Approach)
Compare this to the Day 3 history earlier: the manual approach needed 4 separate statements (plus auto-OPTIMIZE noise). This clone's history should show the DEEP CLONE as version 0, and this MERGE as a single version 1 — one transaction covering everything.

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_merge_demo`;

## Verify MERGE Result
Should match the manual Day 3 end state exactly: eno 1,2,3,5,6, with eno=3 and eno=5 updated, eno=4 gone, eno=6 new.

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_merge_demo` ORDER BY eno;

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;